# L3 — Tool Use

> **El modelo decide cuándo invocar herramientas externas.** Por primera vez, el flujo no lo controla solo el código — el modelo participa.

## ¿Qué vas a aprender?

- Cómo definir **tools** para que el modelo las use
- El ciclo `tool_use` ↔ `tool_result` y por qué hay que escribir un bucle
- Cómo el modelo **decide** qué tool invocar y cuándo
- Diferencia con L2: del control total al control compartido

## ¿Qué es L3?

En L2 el modelo trabaja ciego: solo sabe lo que le pasas en el prompt. En L3 le das acceso a **herramientas** — funciones que puede invocar para obtener información externa o ejecutar acciones.

**La diferencia fundamental con L2:** el modelo participa en el flujo. Tú defines las herramientas disponibles, pero **él decide cuándo usarlas y con qué parámetros**.

## Cómo funciona el ciclo

El ciclo de una llamada con tool use tiene **4 pasos**:

```
1. Tú defines las tools disponibles y envías el mensaje
2. El modelo decide si necesita una tool y devuelve un bloque tool_use (no texto)
3. Tú ejecutas la función real con los parámetros que dio el modelo
4. Devuelves el resultado al modelo y él genera la respuesta final
```

**El modelo nunca ejecuta código directamente.** Dice *"quiero llamar a `search_similar_tickets` con `query='auth 500 error'`"* — y tú ejecutas esa llamada real y le devuelves el resultado. **El control de seguridad siempre está en tu código.**

## L2 vs L3 — comparativa

|  | **L2 — Chain** | **L3 — Tool use** |
|---|------------|---------------|
| Flujo | Lo controla el código siempre | El modelo decide si invocar tools |
| Información disponible | Solo lo que está en el prompt | Puede consultar sistemas externos |
| Determinismo | Alto — siempre los mismos pasos | Medio — el modelo elige qué tools usar |
| Complejidad | Baja | Media |

## Conceptos clave

### Tool definition
Antes de la llamada le describes al modelo las tools: nombre, descripción y parámetros con sus tipos. **El modelo usa esa descripción para decidir cuándo invocarla** — por eso la descripción importa tanto como el código.

### Tool use block
Cuando el modelo decide invocar una tool, no devuelve texto — devuelve un bloque de tipo `tool_use`:

```json
{
  "type": "tool_use",
  "name": "search_similar_tickets",
  "input": { "query": "authentication service 500 error", "max_results": 3 }
}
```

### Tool result
El resultado de la tool vuelve al modelo como un mensaje de tipo `tool_result`. El modelo lo lee y decide si necesita más tools o si ya tiene suficiente para responder.

### Stop reason
En L2 el modelo siempre terminaba con `stop_reason: "end_turn"`. En L3 puede terminar con `"tool_use"` — significa que quiere invocar una tool y está esperando el resultado. **Tu código tiene que manejar ambos casos.**

---
## Setup — Imports y `parse_json_safe`

Cargamos el cliente Anthropic y la utilidad de parsing JSON (igual que en L2).

> **Requisitos:** `pip install anthropic python-dotenv`

In [ ]:
"""
L3 — Tool use: clasificador de tickets con herramientas externas
================================================================
El modelo decide cuándo invocar las tools disponibles.
El flujo ya no es 100% nuestro — el modelo participa en él.

Tools disponibles:
    - search_similar_tickets: busca tickets parecidos resueltos anteriormente
    - get_system_status: consulta el estado actual de los servicios

Las tools son mocks — devuelven datos simulados.
Cuando tengas la integración real, solo cambias la implementación
de la tool, no el resto del sistema.

Requisitos:
    pip install anthropic

Variables de entorno:
    ANTHROPIC_API_KEY
"""

from pathlib import Path
from dotenv import load_dotenv
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / ".env").exists():
        load_dotenv(_p / ".env", override=True)
        break

import os
import json
import anthropic

client = anthropic.Anthropic()
MODEL  = "claude-sonnet-4-5"

## Definición de las tools

Declaramos las dos tools disponibles para el modelo:

- **`search_similar_tickets`** — historial de tickets resueltos (contexto pasado)
- **`get_system_status`** — estado actual de un servicio (contexto presente)

> **Importante:** la `description` de cada tool es **el contrato que el modelo lee**. Una descripción vaga = el modelo no sabe cuándo usarla. Una descripción precisa = la usa en el momento correcto.

In [ ]:
# ─────────────────────────────────────────────
# Definición de tools
#
# El modelo usa la "description" para decidir cuándo invocar cada tool.
# Una descripción vaga = el modelo no sabe cuándo usarla.
# Una descripción precisa = el modelo la usa en el momento correcto.
# ─────────────────────────────────────────────

def parse_json_safe(text: str) -> dict:
    """
    Parsea JSON de forma segura.
    Los LLMs a veces envuelven la respuesta en ```json ... ```.
    """
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.split("\n")
        cleaned = "\n".join(lines[1:-1])
    return json.loads(cleaned)

TOOLS = [
    {
        "name": "search_similar_tickets",
        "description": (
            "Search for similar support tickets that have been resolved in the past. "
            "Use this when you need historical context about a problem — "
            "for example, to check if this issue has occurred before and how it was resolved."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Short description of the problem to search for"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to return (default: 3)"
                }
            },
            "required": ["query"]
        }
    },
    {
        "name": "get_system_status",
        "description": (
            "Get the current status of a service. Use this ONLY when the ticket explicitly mentions a service being down or returning errors in production. Do not use for UI or display issues."
            "Use this when the ticket mentions a service being down or degraded, "
            "to check if there is an active incident before classifying severity."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "service": {
                    "type": "string",
                    "description": "Name of the service to check (e.g. 'auth', 'payments', 'api')"
                }
            },
            "required": ["service"]
        }
    }
]

## Implementación de las tools (mocks)

Las funciones reales que se ejecutan cuando el modelo invoca una tool. Aquí están como **mocks** que devuelven datos simulados — en producción harían llamadas reales a Jira, PagerDuty, tu sistema de monitorización, etc.

`execute_tool` es el **dispatcher**: el modelo dice qué quiere ejecutar y nosotros decidimos si y cómo ejecutarlo. Es el punto de control de seguridad — aquí podrías añadir validación, rate limiting, logging, autorización...

In [ ]:
# ─────────────────────────────────────────────
# Implementación de tools (mocks)
#
# En producción aquí iría la llamada real a Jira, PagerDuty, etc.
# La interfaz del resto del sistema no cambia.
# ─────────────────────────────────────────────

def search_similar_tickets(query: str, max_results: int = 3) -> list[dict]:
    """Mock: simula una búsqueda en el historial de tickets."""
    mock_tickets = [
        {
            "id": "TICKET-1042",
            "title": "Authentication service returning 500 on login",
            "resolution": "Database connection pool exhausted. Increased max connections and restarted the service.",
            "severity": "P1",
            "resolved_in_minutes": 23
        },
        {
            "id": "TICKET-987",
            "title": "Users unable to log in after deploy",
            "resolution": "Bad environment variable in the new deploy. Rolled back and fixed the config.",
            "severity": "P1",
            "resolved_in_minutes": 45
        },
        {
            "id": "TICKET-756",
            "title": "Intermittent 500 errors on auth endpoint",
            "resolution": "Memory leak in the token validation middleware. Patched and deployed hotfix.",
            "severity": "P2",
            "resolved_in_minutes": 90
        },
    ]
    return mock_tickets[:max_results]


def get_system_status(service: str) -> dict:
    """Mock: simula una consulta al estado de los servicios."""
    mock_statuses = {
        "auth": {
            "status": "degraded",
            "active_incident": True,
            "incident_id": "INC-2024-089",
            "started_at": "2024-01-15T09:15:00Z",
            "affected_regions": ["eu-west-1"]
        },
        "payments": {
            "status": "operational",
            "active_incident": False
        },
        "api": {
            "status": "operational",
            "active_incident": False
        }
    }
    return mock_statuses.get(service, {"status": "unknown", "active_incident": False})


def execute_tool(tool_name: str, tool_input: dict) -> str:
    """
    Ejecuta la tool real y devuelve el resultado como string JSON.

    Este es el punto de control de seguridad: el modelo dice qué
    quiere ejecutar, pero nosotros decidimos si ejecutarlo.
    Aquí podrías añadir validación, rate limiting, logging, etc.
    """
    print(f"  [Tool] Ejecutando: {tool_name}({tool_input})")

    if tool_name == "search_similar_tickets":
        result = search_similar_tickets(**tool_input)
    elif tool_name == "get_system_status":
        result = get_system_status(**tool_input)
    else:
        result = {"error": f"Tool '{tool_name}' no encontrada"}

    return json.dumps(result, ensure_ascii=False)

## El agente con tool use

El **system prompt** define cómo el modelo debe usar las tools y qué formato JSON devolver.

`run_agent` implementa el **bucle de tool use**:
1. Llama al modelo con las tools disponibles
2. Si `stop_reason == "end_turn"` → respuesta final, parseamos y devolvemos
3. Si `stop_reason == "tool_use"` → ejecutamos las tools que pidió y volvemos al paso 1

A diferencia de L2, **el flujo no es lineal** — el modelo puede invocar tools en cualquier orden y tantas veces como necesite.

In [ ]:
# ─────────────────────────────────────────────
# Agente con tool use
#
# A diferencia de L2, aquí el flujo no es lineal.
# El modelo puede invocar tools en cualquier orden y
# tantas veces como necesite antes de responder.
# ─────────────────────────────────────────────

SYSTEM_PROMPT = """
You are a senior support engineer that classifies and triages support tickets.

You have access to tools to help you make better decisions:
- Use search_similar_tickets to check if this problem has occurred before
- Use get_system_status to check if there is an active incident for the affected service

Always use the available tools before classifying a ticket — historical context and
current system status are critical for an accurate severity assessment.

Respond ONLY with valid JSON. No additional text.
Schema:
{
  "severity": "P1" | "P2" | "P3" | "P4",
  "reason": string,
  "area": "backend" | "frontend" | "infra" | "data" | "security",
  "requires_escalation": boolean,
  "similar_incidents": [string],
  "recommended_action": string
}
""".strip()


def run_agent(ticket: str) -> dict:
    """
    Ejecuta el agente con tool use.

    El bucle continúa mientras el modelo quiera invocar tools
    (stop_reason == "tool_use"). Cuando termina (stop_reason == "end_turn")
    parseamos la respuesta final.
    """
    print("=" * 60)
    print("TICKET:", ticket[:80] + "..." if len(ticket) > 80 else ticket)
    print("=" * 60)

    messages = [{"role": "user", "content": f"[USER]\n{ticket}\n[/USER]"}]

    # Bucle de tool use — el modelo puede invocar varias tools
    # antes de dar la respuesta final
    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            temperature=0,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages
        )

        print(f"\n[Modelo] stop_reason: {response.stop_reason}")

        # El modelo ha terminado — respuesta final
        if response.stop_reason == "end_turn":
            text = next((b.text for b in response.content if b.type == "text"), "")
            print("DEBUG raw:", repr(text))  # añade esto
            result = parse_json_safe(text)
            print("\n" + "=" * 60)
            print("RESULTADO FINAL:")
            print(json.dumps(result, ensure_ascii=False, indent=2))
            return result

        # El modelo quiere invocar una o más tools
        if response.stop_reason == "tool_use":

            # Añadimos la respuesta del modelo al historial
            messages.append({"role": "assistant", "content": response.content})

            # Procesamos todos los bloques tool_use de la respuesta
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })

            # Devolvemos los resultados al modelo y continuamos el bucle
            messages.append({"role": "user", "content": tool_results})

## Tickets de ejemplo

Dos casos de prueba para ver el comportamiento del modelo:

- **`ticket_critical`** — el modelo debería invocar **ambas** tools (system status + tickets similares) antes de clasificar como P1
- **`ticket_minor`** — bug visual; el modelo debería usar pocas o ninguna tool y clasificar como P3/P4

In [ ]:
# ─────────────────────────────────────────────
# Ejemplos de uso
# ─────────────────────────────────────────────

ticket_critical = """
Users cannot log in since 09:15 UTC. The authentication service is returning
500 errors on POST /auth/login. All environments affected. We are seeing
database connection timeouts in the logs. Approximately 3,000 users impacted.
""".strip()

ticket_minor = """
On the account settings page, when the user updates their email address,
the confirmation message shows the old email instead of the new one.
The update itself works correctly — it is only a display issue.
Reported in production, low impact.
""".strip()

## Ejecutar — caso crítico

Lanza el agente sobre el ticket crítico. Observa cómo el modelo invoca las tools (verás `[Tool] Ejecutando: ...`) antes de devolver el JSON final:

In [ ]:
run_agent(ticket_critical)

Ahora el caso menor — fíjate en si el modelo decide usar tools o no, y por qué:

In [ ]:
run_agent(ticket_minor)

## Siguientes pasos → L4

Con tool use el modelo puede **consultar información externa**, pero solo si tú le defines la tool exacta. Para conocimiento que cambia constantemente (documentación, runbooks, etc.) eso no escala.

En **[L4 — RAG](../L4-RAG/L4_RAG.ipynb)** veremos cómo dar al modelo acceso a una **base de conocimiento** mediante búsqueda semántica.